# Lab 12 — HR Attrition Mini Case Study
**Course Capstone Track** · Intermediate Capstone · ~75 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute overall attrition and slice by Department / OverTime
2. Bin income into quartiles and compare attrition rates
3. Spot confounders and non-monotonic signals
4. Write 3 actionable hypotheses into attrition_report.md

## Datasets (this folder)
- `HR-Employee-Attrition-synth.csv` — auto-download from `https://raw.githubusercontent.com/aaubs/ds-master/main/apps/M1-attrition-streamlit/HR-Employee-Attrition-synth.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-12-hr-attrition-capstone/lab-12-hr-attrition-capstone.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-12-hr-attrition-capstone"
# Hosted manifest (matheshcp/ai_course_content, branch main).
MANIFEST_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-12-hr-attrition-capstone/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("HR-Employee-Attrition-synth.csv", "https://raw.githubusercontent.com/aaubs/ds-master/main/apps/M1-attrition-streamlit/HR-Employee-Attrition-synth.csv")
    _ensure_file("emp_attrition.csv", "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## Course Capstone Track: EDA → Insight → Executive One-Pager

> **Scenario:** People analytics shares `HR-Employee-Attrition-synth.csv` (2000 rows, IBM-style HR schema, 35 columns). Deliver an executive one-pager: overall attrition rate, rates by Department and OverTime, income quartiles vs attrition, and **3 actionable hypotheses** with supporting numbers. Write `attrition_report.md`.
>
> **You will learn:** end-to-end pipeline, EDA → insight → narrative, avoiding spurious correlations.
> **Time:** ~75 minutes. **Level:** Intermediate Capstone. **Needs:** pandas + matplotlib. **Env:** 🟢 Colab only.

### Capstone mental map

| Stage | Question | Artifact |
|---|---|---|
| Frame | What is attrition rate? | headline number |
| Slice | Where is it worst? | Dept / OverTime tables |
| Associate | Income vs leaving? | quartile rates + chart |
| Interpret | Causation? confounders? | caveats section |
| Act | What would we try? | 3 hypotheses |

---

### 1. Load and profile (local first, Colab fallback)

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def load_hr():
    local = "HR-Employee-Attrition-synth.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/aaubs/ds-master/main/apps/"
            "M1-attrition-streamlit/HR-Employee-Attrition-synth.csv",
            local,
        )
    df = pd.read_csv(local)
    if str(df.columns[0]).startswith("Unnamed"):
        df = df.drop(columns=df.columns[0])
    return df

df = load_hr()
print(df.shape)          # (2000, 35)
print(df["Attrition"].value_counts())
# No 1702, Yes 298
print("overall attrition rate:", round((df["Attrition"] == "Yes").mean(), 4))
# 0.149
print(df["Department"].value_counts())
# Research & Development 1098, Sales 799, Human Resources 103
print("mean age:", df["Age"].mean().round(2))  # 37.24


---

### 2. Headline: attrition by Department and OverTime

In [ ]:
def rate(col):
    return (df.groupby(col)["Attrition"]
              .apply(lambda s: (s == "Yes").mean())
              .sort_values(ascending=False)
              .round(4))

print(rate("Department"))
# Research & Development    0.1621
# Sales                     0.1377
# Human Resources           0.0971

print(rate("OverTime"))
# Yes    0.1869
# No     0.1387


**OverTime gap:** 18.7% vs 13.9% — **+4.8 pp** for staff working overtime.

In [ ]:
ax = rate("OverTime").plot(kind="bar", figsize=(5, 3), rot=0, ylim=(0, 0.25),
                           title="Attrition rate by OverTime")
ax.set_ylabel("Attrition rate")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
fig = ax.get_figure(); fig.tight_layout(); fig.savefig("attrition_overtime.png", dpi=120)


---

### 3. Income quartiles vs attrition

In [ ]:
df["IncomeQ"] = pd.qcut(df["MonthlyIncome"], 4,
                        labels=["Q1", "Q2", "Q3", "Q4"])
edges = df["MonthlyIncome"].quantile([0, .25, .5, .75, 1]).tolist()
print("quartile edges:", [round(e, 2) for e in edges])
# [1009.0, 5040.75, 6776.0, 9667.25, 28723.0]

by_q = df.groupby("IncomeQ", observed=True)["Attrition"].apply(
    lambda s: (s == "Yes").mean()).round(4)
print(by_q)
# Q1 0.1440
# Q2 0.1238
# Q3 0.1503
# Q4 0.1780


**Counter-intuitive (synthetic) pattern:** highest quartile has the *highest* attrition (17.8%) — in this synthetic file, not the classic IBM original. Report what you see; don’t force the textbook story.

In [ ]:
ax = by_q.plot(kind="bar", figsize=(5, 3), rot=0, ylim=(0, 0.25),
               title="Attrition rate by income quartile")
ax.set_ylabel("Attrition rate")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
for i, v in enumerate(by_q):
    ax.text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=9)
fig = ax.get_figure(); fig.tight_layout()
fig.savefig("attrition_income_quartiles.png", dpi=120)
print("saved charts")


---

### 4. Cross-checks (confounders)

In [ ]:
# Overtime × income — is the OT gap just a pay effect?
print(pd.crosstab(df["OverTime"], df["IncomeQ"], normalize="index").round(3))

# Job satisfaction vs attrition (weak in this synth?)
print(rate("JobSatisfaction"))
# 4: 0.1576, 2: 0.1565, 1: 0.1383, 3: 0.1254  (non-monotonic — weak signal)

# Mean income by leaver status
print(df.groupby("Attrition")["MonthlyIncome"].mean().round(2))
# No 8033.23, Yes 8676.02  (leavers earn slightly more here)


> **Spurious-correlation guard:** always ask “what else varies with this feature?” Department, job level, and overtime often proxy for each other. n = 103 in HR dept → wide uncertainty on that 9.7% rate.

---

### 5. Draft the executive one-pager

In [ ]:
report = f"""# Attrition one-pager — HR synthetic (n=2000)

## Headline
- Overall attrition rate: **{(df['Attrition']=='Yes').mean():.1%}** (298 leavers / 2000).
- Mean age {df['Age'].mean():.1f}; majority in R&D ({(df['Department']=='Research & Development').mean():.0%}).

## Where attrition concentrates
| Cut | Higher rate | Lower rate | Gap |
|---|---|---|---|
| OverTime | Yes {(df.loc[df.OverTime=='Yes','Attrition']=='Yes').mean():.1%} | No {(df.loc[df.OverTime=='No','Attrition']=='Yes').mean():.1%} | +4.8 pp |
| Department | R&D 16.2% | HR 9.7% (n=103, noisy) | — |
| Income quartile | Q4 17.8% | Q2 12.4% | +5.4 pp |

## Three hypotheses (to test, not truths)
1. **Overtime load raises exit risk.** Design: compare matched OT/non-OT pairs within JobLevel; pilot workload cap in R&D.
2. **High earners in this population leave for external offers** (Q4 17.8%). Design: exit-interview coding + comp-ratio analysis vs market.
3. **R&D career-path opacity drives exits** (dept rate 16.2%). Design: promotion velocity by tenure; mentoring experiment in two teams.

## Caveats
- Synthetic data; JobSatisfaction signal is non-monotonic — do not over-interpret.
- Department rates with n≈100 have wide CIs; show counts beside every percentage.
- Association ≠ causation; validate with interventions before policy change.
"""
open("attrition_report.md", "w", encoding="utf-8").write(report)
print("wrote attrition_report.md")
print(report[:400])


---

## Exercises (do these!)

### Exercise 1 — Attrition by OverTime
Print attrition rate for `OverTime == "Yes"` and `"No"` (4 d.p.). What is the percentage-point gap?
*Expected: Yes 0.1869 · No 0.1387 · gap ≈ 0.0482 (4.8 pp).*

<details>
<summary>Hint</summary>

`df.groupby("OverTime")["Attrition"].apply(lambda s: (s=="Yes").mean())`.
</details>

### Exercise 2 — Income quartiles vs attrition
`pd.qcut(MonthlyIncome, 4)` → attrition rate per quartile. Print the 4 rates and the quartile edges.
*Expected: Q1 0.1440 · Q2 0.1238 · Q3 0.1503 · Q4 0.1780 · edges 1009, 5040.75, 6776, 9667.25, 28723.*

<details>
<summary>Hint</summary>

`observed=True` on groupby avoids empty-group warnings in newer pandas.
</details>

### Exercise 3 — Draft 3 hypotheses with numbers
Write three hypotheses; each must cite at least one number from this lab and one validation step. Save under `attrition_report.md`.
*Expected: free-form — e.g. OT pilot, R&D career path, comp-ratio for Q4 leavers — each with rates/n from Sections 2–4.*

<details>
<summary>Hint</summary>

Template: “If we [intervention], then [metric] changes, because [evidence: X% vs Y%]; validate via [design].”
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
rates = df.groupby("OverTime")["Attrition"].apply(lambda s: (s == "Yes").mean())
print(rates.round(4))
gap = rates["Yes"] - rates["No"]
print(f"gap = {gap:.4f} ({gap*100:.1f} pp)")
# Yes    0.1869
# No     0.1387
# gap = 0.0482 (4.8 pp)

# --- Solution 2 ---
df["IncomeQ"] = pd.qcut(df["MonthlyIncome"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df.groupby("IncomeQ", observed=True)["Attrition"]
      .apply(lambda s: (s == "Yes").mean()).round(4))
print(df["MonthlyIncome"].quantile([0, .25, .5, .75, 1]).tolist())
# Q1 0.1440  Q2 0.1238  Q3 0.1503  Q4 0.1780
# [1009.0, 5040.75, 6776.0, 9667.25, 28723.0]

# --- Solution 3 ---
# Template (fill with your own numbers from above):
hypotheses = """
1. Reducing mandatory overtime in R&D will cut attrition — OT yes 18.7% vs no 13.9%.
2. Q4 earners need external-market comp review — Q4 attrition 17.8% vs Q2 12.4%.
3. Promotion clarity in R&D should be audited — dept rate 16.2% (n=1098).
Validate: matched cohorts / pilot teams / exit-interview themes; not one-shot correlations.
"""
print(hypotheses)
# (Section 5 already writes attrition_report.md with this structure)


### What to learn next
- Logistic regression / SHAP for driver ranking (Course 2).
- Survival analysis for time-to-exit.
- Fairness: would an OT-based policy disproportionately hit caregiver demographics?
- Cheat sheet: headline rate → slice → quartile/associate → confounders → hypotheses with validation plans.

*Files in this folder: `HR-Employee-Attrition-synth.csv` (primary), `emp_attrition.csv` (IBM alternate, n=1470) · outputs `attrition_overtime.png`, `attrition_income_quartiles.png`, `attrition_report.md`.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
